In [2]:
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from wfcrl.rewards import StepPercentage
from wfcrl import environments as envs

from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

import torch

import warnings
warnings.filterwarning("ignore")

sns.set_theme(style="whitegrid")
SEED = 13
N_sim = 1000 # 100_000

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [5]:
def step_policy(i):
    '''Randmoly samples actions'''
    joint_action = {"yaw": np.zeros(env.num_turbines)}
    mask = np.random.random(env.num_turbines) < 0.3 # proba of a turbine to get controled
    joint_action["yaw"][mask] = np.random.uniform(-40, 40, size=mask.sum())
    return joint_action

def obs_to_row(obs, reward, power, step):
    '''Writes all episode results into a dict'''
    row = {"step": step, "reward": reward[0]}
    for key, val in obs.items():
        val = np.atleast_1d(val)
        if len(val) == 1:
            row[key] = val[0]
        else:
            for i, v in enumerate(val):
                row[f"{key}_{i}"] = v
    for i in range(len(power)):
        row[f"power_{i}"] = power[i]
    return row

In [4]:
env = envs.make("Turb3_Row1_Floris", 
                max_num_steps=100,
                controls={"yaw": (-40, 40)},
                continuous_control = False, # continuous action space
                log=True)

/home/aleksei/Documents/Research/GraphPFNs/weather-vein/wfcrl-env/wfcrl/mdp.py:203: UserWarning: No step size was provided for actuator yaw. Step size will default to 1.
  warn(


In [6]:
sims, max_reward, rows = 0, -np.inf, []
while len(rows) < N_sim:
    observation = env.reset(seed=SEED) # to fix initial coniditions (Sc.1): options={"wind_speed": 8, "wind_direction": 270}
    r, i = 0, 0
    done = False
    while not done:
        joint_action = step_policy(i)
        observation, reward, termination, truncation, info = env.step(joint_action)
        rows.append(obs_to_row(observation, reward, info["power"], i))  
        r += reward
        i += 1
        done = termination or truncation
    if sims%500==0:
        print(f"Simulation #{sims}: total reward = {r}")
    sims += 1 
    max_reward = max(max_reward, r)
print(f"Maximum total reward: {max_reward}")
sim_summary = pd.DataFrame(rows)
sim_summary

Simulation #0: total reward = [278.03150444]
Maximum total reward: [278.03150444]


,step,reward,yaw_0,yaw_1,yaw_2,freewind_measurements_0,freewind_measurements_1,wind_speed_0,wind_speed_1,wind_speed_2,wind_direction_0,wind_direction_1,wind_direction_2,power_0,power_1,power_2
0,0,2.922892,-1.000000,36.259937,-1.000000,8.555233,208.433362,8.527036,8.527036,8.527036,208.552207,211.840310,208.699327,2.067434,1.381572,2.067434
1,1,2.937855,-2.000000,35.259937,-2.000000,8.555233,208.433362,8.527036,8.527036,8.527036,208.407831,211.840524,208.554011,2.065563,1.413470,2.065563
2,2,2.952758,-3.000000,34.259937,-3.000000,8.555233,208.433362,8.527036,8.527036,8.527036,208.263965,211.834898,208.408980,2.062444,1.447760,2.062444
3,3,2.763228,24.564875,33.259937,10.387836,8.555233,208.433362,8.527036,8.527036,8.527036,211.512652,211.950040,210.310229,1.730848,1.481036,2.005955
4,4,2.800530,23.564875,32.259937,9.387836,8.555233,208.433362,8.527036,8.527036,8.527036,211.448871,211.935718,210.180726,1.757618,1.513299,2.016913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1084,94,2.673389,-24.445442,-40.000000,-3.985611,8.555233,208.433362,8.527036,8.527036,8.527036,205.828546,205.420707,208.003364,1.734105,1.256818,2.058148
1085,95,2.655640,-25.445442,-40.000000,-4.985611,8.555233,208.433362,8.527036,8.527036,8.527036,205.765309,205.418159,207.860801,1.706438,1.256818,2.052551
1086,96,2.638507,-26.445442,-40.000000,-5.985611,8.555233,208.433362,8.527036,8.527036,8.527036,205.707787,205.415823,207.719714,1.680996,1.256818,2.045889
1087,97,2.620565,-27.445442,-40.000000,-6.985611,8.555233,208.433362,8.527036,8.527036,8.527036,205.656053,205.413702,207.580402,1.654560,1.256818,2.038704


In [7]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [8]:
X = sim_summary[['yaw_0', 'yaw_1', 'yaw_2', 'freewind_measurements_0', 'freewind_measurements_1']]
y = sim_summary[['power_0', 'power_1', 'power_2']]

In [11]:
y_train

,power_0,power_1,power_2
176,1.760835,1.256818,1.256818
18,2.015054,1.880884,1.783274
764,1.613439,1.256818,1.989579
696,2.058076,2.023769,2.058076
61,1.847326,1.256818,1.256818
...,...,...,...
330,1.339040,1.530534,1.407888
466,1.256818,1.696503,1.256818
121,1.992957,1.523145,1.995013
1044,1.256818,1.623747,1.963854


In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)
models = []
for i in range(3):
    regressor = TabPFNRegressor(random_state=SEED,)  
    regressor.fit(X_train, y_train[f'power_{i}'])
    models.append(regressor)

/home/aleksei/Documents/Research/GraphPFNs/weather-vein/.venv/lib/python3.12/site-packages/tabpfn/validation.py:56: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(
/home/aleksei/Documents/Research/GraphPFNs/weather-vein/.venv/lib/python3.12/site-packages/tabpfn/validation.py:56: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(
/home/aleksei/Documents/Research/GraphPFNs/weather-vein/.venv/lib/python3.12/site-packages/tabpfn/validation.py:56: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


In [13]:
freewind = torch.tensor(observation['freewind_measurements'], dtype=torch.double)
current_yaw = observation['yaw']

yaws = torch.tensor(current_yaw, dtype=torch.double).requires_grad_(True)

In [26]:
test_X = torch.cat([yaws, freewind]).reshape(1,-1)
test_X.shape

torch.Size([1, 5])

/home/aleksei/Documents/Research/GraphPFNs/weather-vein/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but TabPFNRegressor was fitted with feature names
  warnings.warn(


array([2.029794], dtype=float32)

In [31]:
observation = env.reset(seed=SEED)
r, i = 0, 0
done = False
while not done:
    '''
    The goal is to optimize the yaw given the trained GP which provides PPD over power given yaws.
    Therefore, having GP approximating the turbine physics we look for the most optimal action without taking any decisions in the 
    simulation itself.
    '''
    freewind = torch.tensor(observation['freewind_measurements'], dtype=torch.double)
    current_yaw = observation['yaw']
    
    yaws = torch.tensor(current_yaw, dtype=torch.double).requires_grad_(True)
    optimizer = torch.optim.Adam([yaws], lr=1.0) # lr is high since the error surface is smooth and dim=3
    
    for _ in range(100):
        optimizer.zero_grad()
        test_X = torch.cat([yaws, freewind]).reshape(1,-1)
        pred_power = 0
        for i in range(3):
            pred_power += models[i].predict(test_X.detach().numpy())[0]
        (-torch.tensor(pred_power).requires_grad_(True)).backward()  # negative for maximization
        optimizer.step()
        yaws.data.clamp_(-40, 40)
    
    joint_action = {'yaw': yaws.detach().numpy() - current_yaw}
    
    observation, reward, termination, truncation, info = env.step(joint_action)
    r += reward
    i += 1
    done = termination or truncation

print(f"Total reward = {r}")

/home/aleksei/Documents/Research/GraphPFNs/weather-vein/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but TabPFNRegressor was fitted with feature names
  warnings.warn(
/home/aleksei/Documents/Research/GraphPFNs/weather-vein/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but TabPFNRegressor was fitted with feature names
  warnings.warn(
/home/aleksei/Documents/Research/GraphPFNs/weather-vein/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but TabPFNRegressor was fitted with feature names
  warnings.warn(
/home/aleksei/Documents/Research/GraphPFNs/weather-vein/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but TabPFNRegressor was fitted with feature names
  warnings.warn(
/home/aleksei/Documents/Research

KeyboardInterrupt: 